# **Rebuild posts file (one-time)**

For a topic run made **before** `topic_modelling` wrote `posts_<ISSUES>.parquet`. It rebuilds that file from the raw
exports and the run's `documents_<ISSUES>.parquet`, so the network notebook can use the run without refitting.

```
raw exports → load and merge → date normalization → format normalization → map topic id → posts_<ISSUES>.parquet
```

The result is saved next to the run's other files, under the name `network_analysis.ipynb` expects for `POSTS_FILE`.
Use the **same raw files and the same keyword settings** as the topic run: the topic of a post comes from matching
its exact text to the run's documents, and the last cell checks how many texts were found.

### **Mounting Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

### **Configuration**
Copy `PATH`, `FILE_NAME`, `COLUMNS`, `SENTIMENT_MAP` and the keyword settings from the topic run's config cell.

In [ ]:
ISSUES          = 'Putusan MK Gibran' # the topic run's ISSUES
TOPIC_DIR       = '/content/drive/MyDrive/model_result/Putusan MK Gibran_07102026' # that run's OUT_DIR: holds documents_<ISSUES>.parquet
PATH            = '/content/drive/MyDrive/Data/Raw Data Prabowo 2026' # raw exports the run read
FILE_NAME       = 'prabowo26_oct*' # one file or a pattern, as in the run
COLUMNS = {
    'Date':      'Date',        # e.g. 'creation_time'
    'Media':     'Media',       # e.g. 'specific_resource_type'
    'Author':    'Author',      # e.g. 'from_username'
    'Mentions':  'Mentions',    # e.g. 'content'
    'Sentiment': 'Sentiment',   # e.g. 'final_sentiment'
    'Link':      'Link',
}
SENTIMENT_MAP   = {} # e.g. {'mixed': 'neutral'}
KEYWORDS        = []   # the run's keyword filter; [] or None -> no filtering
REQUIRED        = []   # must-have terms (ALL of them); [] -> none
MIN_HITS        = 2    # how many of KEYWORDS a row must match
EXACT_WORD      = False

### **Notebook Set Up**

In [ ]:
%pip install -q "great[all] @ git+https://github.com/azmkto/GREAT-Tools.git"

import glob, os, re

import pandas as pd

from great import MEDIA_MAP, plat_class, sent_class

### **1. Load and merge raw data**
Each file's header row is found by looking for every column named in `COLUMNS` (exports sometimes carry a title line
above it). Every column is kept; the `COLUMNS` ones are renamed and put first. Rows that appear in two files (identical
in every column, e.g. overlapping exports) are kept once.

In [ ]:
SEPARATORS = ';,\t'


def is_csv(f):
    return f.lower().endswith('.csv')


def header_row(f):
    # First row (within the top 20) whose cells include every column named in COLUMNS.
    want = set(COLUMNS.values())
    if is_csv(f):
        with open(f, encoding='utf-8-sig', errors='replace') as fh:
            rows = [next(fh, '') for _ in range(20)]
        cells = [{c.strip().strip('"') for s in SEPARATORS for c in line.split(s)} for line in rows]
    else:
        top = pd.read_excel(f, header=None, nrows=20)
        cells = [set(row.astype(str).str.strip()) for _, row in top.iterrows()]
    hits = [i for i, c in enumerate(cells) if want <= c]
    assert hits, f'{f}: no header row with all of {sorted(want)} -- check COLUMNS'
    return hits[0]


def read(f):
    h = header_row(f)
    df = (pd.read_csv(f, skiprows=h, sep=None, engine='python', encoding_errors='replace')
          if is_csv(f) else pd.read_excel(f, header=h))
    df.columns = df.columns.astype(str).str.strip()
    df = df.rename(columns={v: k for k, v in COLUMNS.items()})
    print(f'  {os.path.basename(f)}: {len(df):,} rows')
    return df[list(COLUMNS) + [c for c in df.columns if c not in COLUMNS]]


files = sorted(f for f in glob.glob(f'{PATH}/{FILE_NAME}') if not os.path.basename(f).startswith('~$'))
assert files, f'No files match {PATH}/{FILE_NAME}'
print(f'Reading {len(files)} file(s):')
raw = pd.concat([read(f) for f in files], ignore_index=True)
n = len(raw)
raw = raw.drop_duplicates().reset_index(drop=True)
print(f'{n:,} rows -> {len(raw):,} after dropping rows repeated across files')

### **2. Date normalization**
ISO text (`2026-10-01 10:30:00`, the monitoring tool's layout) is read as ISO. Anything else decides day/month order
from the data: a second field above 12 can only be a day, so the layout is MM/DD. Timezone-aware stamps become Jakarta
local time. Rows whose date can't be read are dropped and counted.

In [ ]:
def parse_dates(s):
    if not pd.api.types.is_datetime64_any_dtype(s):
        txt = s.astype(str).str.strip()
        if txt[s.notna()].str.match(r'\d{4}-\d{2}-\d{2}').all():
            s = pd.to_datetime(txt, format='ISO8601', errors='coerce')
        else:
            dm = txt.str.extract(r'^(\d{1,2})[/.-](\d{1,2})[/.-]\d{2,4}').astype(float)
            s = pd.to_datetime(txt, format='mixed', dayfirst=not (dm[1] > 12).any(), errors='coerce')
    if s.dt.tz is not None:
        s = s.dt.tz_convert('Asia/Jakarta').dt.tz_localize(None)
    return s


raw['Date'] = parse_dates(raw['Date'])
bad = raw['Date'].isna().sum()
raw = raw.dropna(subset=['Date'])
print(f"{raw['Date'].min():%d %b %Y %H:%M} - {raw['Date'].max():%d %b %Y %H:%M}   ({bad:,} rows with an unreadable date dropped)")

### **3. Format normalization**
Labels are matched to `great`'s vocabulary: `Sentiment` to `sent_class` (aliases like *positif*, *neg*, *netral*, plus
`SENTIMENT_MAP`), `Media` to `plat_class` through `MEDIA_MAP`, any capitalization. An unknown label stops the notebook
and names itself, so it can be added to `SENTIMENT_MAP` / `MEDIA_MAP`. Text columns are stripped, count columns become
numbers, rows without text are dropped, and the run's keyword filter is applied.

In [ ]:
SENT_ALIASES = {'pos': 'positive', 'positif': 'positive', 'neg': 'negative',
                'negatif': 'negative', 'neu': 'neutral', 'netral': 'neutral'}


def canon(col, vocab, aliases, setting):
    # Map each label to the vocab entry it names: exact (any case) first, then via an alias.
    lookup = {v.lower(): v for v in vocab}
    aliases = {k.lower(): v.lower() for k, v in aliases.items()}
    resolve = lambda x: lookup.get(str(x).strip().lower()) or lookup.get(aliases.get(str(x).strip().lower(), ''))
    mapping = {x: resolve(x) for x in col.dropna().unique()}
    bad = [x for x, v in mapping.items() if v is None]
    assert not bad, f'Unrecognised {col.name} labels {bad} -- add them to {setting} (valid: {sorted(vocab)})'
    return col.map(mapping)


def filter_by_keywords(df, col='Mentions', keywords=KEYWORDS, min_hits=MIN_HITS,
                       required=REQUIRED, exact_word=EXACT_WORD):
    # Keep rows matching >= min_hits of `keywords` AND all of `required` (the topic run's filter).
    keywords, required = list(keywords or []), list(required or [])
    end = r'\b' if exact_word else ''
    has = {w: df[col].str.contains(rf'\b{re.escape(w)}{end}', case=False, na=False) for w in keywords + required}
    mask = pd.Series(True, index=df.index)
    if keywords:
        mask &= sum(has[w] for w in keywords) >= min(min_hits, len(keywords))
    for w in required:
        mask &= has[w]
    print(f'keyword filter: {len(df):,} -> {int(mask.sum()):,} rows')
    return df[mask]


raw['Sentiment'] = canon(raw['Sentiment'], sent_class, {**SENT_ALIASES, **SENTIMENT_MAP}, 'SENTIMENT_MAP')
raw['Media'] = canon(raw['Media'], plat_class, MEDIA_MAP, 'MEDIA_MAP')
for c in ('Author', 'Link'):
    raw[c] = raw[c].astype('string').str.strip()
for c in ('Followers', 'Retweeted', 'Favourited'):
    if c in raw:
        raw[c] = pd.to_numeric(raw[c], errors='coerce')
raw = raw.dropna(subset=['Mentions'])
raw = filter_by_keywords(raw)
print(raw['Sentiment'].value_counts().to_string(), '\n')
print(raw['Media'].value_counts().to_string())

### **4. Map raw data with topic id**
The run modelled each distinct text once, so every copy of a text takes that text's topic. Texts it never modelled
(fewer than 3 tokens after cleaning) stay empty. Names come from `final_result_topic_model_<ISSUES>.xlsx`; topics past
`MAX_ENRICH` have no name there and keep their id only.

**Check the coverage line.** Nearly every document text should be found among the raw posts; a low share means the raw
files or keyword settings differ from the run's.

In [ ]:
documents = pd.read_parquet(f'{TOPIC_DIR}/documents_{ISSUES}.parquet')
topic_of = documents.drop_duplicates('Mentions').set_index('Mentions')['Topic']
names = pd.read_excel(f'{TOPIC_DIR}/final_result_topic_model_{ISSUES}.xlsx', index_col=0).set_index('Topic')['Topic Name']
names[-1] = 'Outlier'

posts = raw.assign(Topic=raw['Mentions'].map(topic_of).astype('Int64'))
posts['TopicLabel'] = posts['Topic'].map(names)

found = topic_of.index.isin(raw['Mentions']).mean()
print(f'coverage: {found:.1%} of the {len(topic_of):,} document texts found among the raw posts')
print(f'{len(posts):,} posts, {posts.Topic.notna().mean():.1%} with a topic, '
      f'{posts.Topic.nunique()} topics ({posts.TopicLabel.isna().sum() - posts.Topic.isna().sum():,} posts in unnamed topics)')
if found < 0.95:
    print('WARNING: under 95% -- the raw files or keyword settings probably differ from the topic run')

### **5. Save the fixed data**

In [ ]:
text_cols = posts.select_dtypes('object').columns
posts[text_cols] = posts[text_cols].astype('string')   # export columns mixing text and numbers break parquet
out = f'{TOPIC_DIR}/posts_{ISSUES}.parquet'
posts.to_parquet(out, index=False)
print('Saved', out)
print('Set POSTS_FILE in network_analysis.ipynb to this path.')